<a href="https://colab.research.google.com/github/20230535/data-mining/blob/main/13%EC%A3%BC%EC%B0%A8_%EC%9E%90%EB%A3%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [공통 시작 셀] Google Drive 마운트 및 파일 확인

from google.colab import drive
#drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content/type3')  # 학생 드라이브 경로에 맞게 수정

required_files = ['churn_logit1.csv', 'loan_default_logit2.csv']

print("=== 파일 존재 여부 확인 ===")
for f in required_files:
    file_path = DATA_DIR / f
    print(f"{f}: {'존재함' if file_path.exists() else '없음'}")

=== 파일 존재 여부 확인 ===
churn_logit1.csv: 없음
loan_default_logit2.csv: 없음


In [2]:
# [실습 1] churn_logit1.csv 로지스틱 회귀: 이탈 여부와 오즈비 해석

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path('/content/type3')  # 필요시 수정
df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('churn ~ age + usage_hour + complaint_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수표 정리
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})

print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 95% 신뢰구간과 오즈비 기준으로도 보기
conf = model.conf_int() #interval
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])   #odd비는 exp
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 예측확률 및 분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #결과를 정수로

print("\n=== 예측확률 상위 5개 ===")
print(df[['churn', 'pred_prob', 'pred_class']].head())

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    direction = '증가' if coef > 0 else '감소'
    print(f"{var}: 계수={coef:.4f}, p-value={pval:.6f}, 오즈비={or_val:.4f}")
    print(f" -> {var}가 1단위 증가할 때 이탈 odds는 약 {or_val:.4f}배가 되며, 방향은 {direction}입니다.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/type3/churn_logit1.csv'

In [ ]:
# [실습 2] loan_default_logit2.csv 로지스틱 회귀: 분류 성능과 residual deviance

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score
from pathlib import Path

DATA_DIR = Path('/content/type3')  # 필요시 수정
df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('default ~ income + debt_ratio + late_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수 / p-value / 오즈비
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)   #오즈비는 exp
})
print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 로그우도와 residual deviance
llf = model.llf
residual_deviance = -2 * llf

print("\n=== 적합도 지표 ===")
print(f"log-likelihood (llf): {llf:.4f}")
print(f"residual deviance: {residual_deviance:.4f}")

# 예측확률, 예측분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #정수로 변환

# 성능지표
acc = accuracy_score(df['default'], df['pred_class'])
err = 1 - acc

print("\n=== 분류 성능 ===")
print(f"accuracy     : {acc:.4f}")
print(f"error rate   : {err:.4f}")

# 예측확률 일부 확인
print("\n=== 예측확률 상위 10개 ===")
print(df[['default', 'pred_prob', 'pred_class']].head(10))

# 신뢰구간
conf = model.conf_int()
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    print(f"{var}: 계수={coef:.6f}, p-value={pval:.6f}, 오즈비={or_val:.6f}")
